In [ ]:
%load_ext autoreload
%autoreload 2

# Import

In [ ]:
import os
import shutil
import cv2
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.colors as mcolors
from pathlib import Path

import sys
sys.path.insert(0, "/project/lt200264-saiwat/WormProject/code/tool")
from worm_visualize_segmentation import get_full_dataframe

sys.path.insert(0, "/project/lt200264-saiwat/Code/Segmentation")
from mask_boundary import boundary_points_to_mask

sys.path.insert(0, "/project/lt200264-saiwat/WormProject/code/pipeline/")
from worm_cls_api import WormClassifier

# Config

In [ ]:
pd.set_option('display.max_columns', None)

CONF_THRES = 0.5

# Path

In [ ]:
LABELS_PATH = "/project/lt200264-saiwat/WormProject/data/worm-24022026/labels/worm-24022026-labels_current.csv"
ROI_PATH = "/project/lt200264-saiwat/WormProject/data/worm-24022026/processed/cropped/dataset_meta.pkl"
MASKS_DIR = '/project/lt200264-saiwat/WormProject/data/worm-24022026/processed/segmentation_masks'
YOLO_PRED_OUT_PATH = "/scratch/lt200264-saiwat/worm-project/detect_worm_cls_4/cls4_predict.pkl"
YOLO_PT_PATH = "/project/lt200264-saiwat/WormProject/models/yolo_worm_cls_size128.pt"
OUTPUT_DIR = "/scratch/lt200264-saiwat/worm-project/detect_worm_cls_4/misslabel"

# Save prediction

In [ ]:
def save_predict(df, model, save_path='./predict.pkl'):
    df_clean = df.copy()
    df_clean['yolo_is_worm'] = False
    df_clean['yolo_conf'] = 0.0
    df_clean['yolo_class_name'] = ''

    # Iterate per image to run batch classification
    for img_path, data in df_clean.groupby('raw_img_path'):
        img = np.array(Image.open(img_path).convert('RGB'))
        img_h, img_w = img.shape[:2]
        bboxes = data['bbox']
        
        if len(bboxes) == 0:
            continue
    
        # ภาพดำ
        full_mask = np.zeros((img_h, img_w), dtype=np.uint8)
    
        # blackout
        for boundary in data['boundary']:
            obj_mask = boundary_points_to_mask(boundary, (img_h, img_w))
            full_mask = cv2.bitwise_or(full_mask, obj_mask.astype(np.uint8))
    
        # ภาพสีของ mask
        masked_img = cv2.bitwise_and(img, img, mask=full_mask)
            
        # Run classifier
        rl = model.classify_boxes(masked_img, bboxes, pad=0)
        
        # Assign results back to the dataframe using index matching
        for idx, res in zip(data.index, rl): # .index คือ ตัว map ไปที่ row จริงๆ
            df_clean.loc[idx, 'yolo_is_worm'] = res.is_worm
            df_clean.loc[idx, 'yolo_conf'] = res.confidence
            df_clean.loc[idx, 'yolo_class_name'] = res.class_name

    # save dataframe
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    df_clean.to_pickle(str(save_path))

# Show prediction

In [ ]:
def show_prediction(df, start=1, stop=None, target_img=None):
    pred_is_worm_df = df[df['yolo_is_worm'] == True].copy()
    
    if target_img is not None:
        if isinstance(target_img, (tuple, list)):
            start, stop = target_img
        else:
            start = target_img
            stop = target_img
            
    for i, (img_path, data) in enumerate(pred_is_worm_df.groupby('raw_img_path'), start=1):
        if i < start:
            continue
        if stop is not None and i > stop:
            break
            
        n_items = len(data)
        if n_items == 0:
            continue
            
        cols = 6
        rows = int(np.ceil((n_items * 2) / cols))
        
        fig, axes = plt.subplots(nrows=rows, ncols=cols, figsize=(18, 5 * rows))
        if isinstance(axes, np.ndarray):
            axes = axes.flatten()
        else:
            axes = [axes]

        img_name = img_path.split('/')[-1]
        fig.suptitle(f'{'=' * 35} {img_name}: {n_items} items {'=' * 35}', fontsize=18, fontweight='bold')
        
        img = np.array(Image.open(img_path).convert('RGB'))
        img_h, img_w = img.shape[:2]

        for obj_idx, (row_idx, obj_id, bbox, boundary, area, aspect_ratio, predict, conf) in enumerate(zip(data.index, data['idx'], data['bbox'], data['boundary'], data['area'], data['aspect_ratio'], data['yolo_is_worm'], data['yolo_conf'])):
            if predict:
                mask_cmap = mcolors.ListedColormap(['#1F51FF']) 
                bbox_color = '#1F51FF'
                # title_text = 'Worm'
            else:
                mask_cmap = mcolors.ListedColormap(['#FF0000'])
                bbox_color = '#FF0000'
                # title_text = 'Not worm'
                
            x, y, w, h = np.ceil(bbox).astype(int)
            pad = 30
            crop_coords = [max(0, y - pad), y + h + pad, max(0, x - pad), x + w + pad]
            crop_img = img[crop_coords[0]:crop_coords[1], crop_coords[2]:crop_coords[3]]

            start_x = max(0, x - pad)
            start_y = max(0, y - pad)
            rect_x = x - start_x
            rect_y = y - start_y

            # --- รูปที่ 1: ภาพตัดเปล่า (Plain Crop) ---
            ax_plain = axes[obj_idx * 2] # ดึงแกนสำหรับรูปซ้าย
            ax_plain.imshow(crop_img)
            ax_plain.set_title(f'Object index {obj_id} | Data row: {row_idx}', fontsize=12, fontweight='bold')
            ax_plain.axis('off')

            # --- รูปที่ 2: ภาพตัดพร้อม Overlay (Mask & BBox) ---
            ax_overlay = axes[obj_idx * 2 + 1] # ดึงแกนสำหรับรูปขวา
            ax_overlay.imshow(crop_img)
            ax_overlay.set_title(f'Conf: {conf:.4f} | Mask area: {area}\n| Aspect ratio: {aspect_ratio}', fontsize=12, color=bbox_color, fontweight='bold')
            ax_overlay.axis('off')
            
            # draw mask
            mask = boundary_points_to_mask(boundary, (img_h, img_w))
            crop_mask = mask[crop_coords[0]:crop_coords[1], crop_coords[2]:crop_coords[3]]
            mask_overlay = np.ma.masked_where(crop_mask == 0, crop_mask) # ซ่อน background เหลือแค่ mask object
            ax_overlay.imshow(mask_overlay, cmap=mask_cmap, alpha=0.5, interpolation='none') # interpolate = ประมาณค่า

            # draw bbox
            rect = rect = patches.Rectangle((rect_x, rect_y), w, h, linewidth=2, edgecolor=bbox_color, facecolor='none') # ตีกรอบเพิ่มเข้าไปในภาพ
            ax_overlay.add_patch(rect)

        # Turn off any extra empty subplots in the grid
        for extra_idx in range(n_items * 2, len(axes)):
            axes[extra_idx].axis('off')

        plt.tight_layout(rect=[0, 0.1, 1, 0.975]) # [left: start, bottom: start, right: stop, top: stop]
        fig.subplots_adjust(hspace=0.6, wspace=0.3) # ช่องว่างระหว่าง subplot ด้านบน/ข้าง
        plt.show()

# Main

## Dataset

In [ ]:
df = get_full_dataframe(LABELS_PATH, ROI_PATH, MASKS_DIR)
df4 = df[df['Class'] == 4].copy()

## Model

In [ ]:
classifier = WormClassifier(YOLO_PT_PATH)

## Save prediction

In [ ]:
save_predict(df4, classifier, YOLO_PRED_OUT_PATH)

## Load new dataset

In [ ]:
df_pred = pd.read_pickle(YOLO_PRED_OUT_PATH)
pred_is_worm_df = df_pred[(df_pred['yolo_is_worm'] == True) & (df_pred['yolo_conf'] >= CONF_THRES)].copy()

## Show predict results

In [ ]:
# show_prediction(pred_is_worm_df)

## Save output

In [ ]:
output_path = Path(OUTPUT_DIR)
output_path.mkdir(parents=True, exist_ok=True)
    
src_dest_dir = output_path / 'src_npy'
mask_dest_dir = output_path / 'mask_npy'
src_dest_dir.mkdir(exist_ok=True)
mask_dest_dir.mkdir(exist_ok=True)

# copy predict worm npy data
for src, mask in zip(pred_is_worm_df['src_npy_path'], pred_is_worm_df['mask_npy_path']):
    if isinstance(src, str) and os.path.exists(src):
        shutil.copy(src, src_dest_dir)
    
    if isinstance(mask, str) and os.path.exists(mask):
        shutil.copy(mask, mask_dest_dir)

# save text file
pred_is_worm_df['Filename'].to_csv(output_path / 'yolo_pred_is_worm.txt', index=False, header=False)

In [ ]:
model = WormClassifier(YOLO_PT_PATH)

In [ ]:
save_predict(dfc4)

In [ ]:
df_clean = dfc4.copy()
df_clean['yolo_is_worm'] = False
df_clean['yolo_conf'] = 0.0
df_clean['yolo_class_name'] = ''

# Iterate per image to run batch classification
for img_path, data in df_clean.groupby('raw_img_path'):
    img = np.array(Image.open(img_path).convert('RGB'))
    img_h, img_w = img.shape[:2]
    bboxes = data['bbox']
    
    if len(bboxes) == 0:
        continue

    # ภาพดำ
    full_mask = np.zeros((img_h, img_w), dtype=np.uint8)

    # blackout
    for boundary in data['boundary']:
        obj_mask = boundary_points_to_mask(boundary, (img_h, img_w))
        full_mask = cv2.bitwise_or(full_mask, obj_mask.astype(np.uint8))

    # ภาพสีของ mask
    masked_img = cv2.bitwise_and(img, img, mask=full_mask)
        
    # Run classifier
    rl = model.classify_boxes(masked_img, bboxes, pad=0)
    
    # Assign results back to the dataframe using index matching
    for idx, res in zip(data.index, rl): # .index คือ ตัว map ไปที่ row จริงๆ
        df_clean.loc[idx, 'yolo_is_worm'] = res.is_worm
        df_clean.loc[idx, 'yolo_conf'] = res.confidence
        df_clean.loc[idx, 'yolo_class_name'] = res.class_name

In [ ]:
pred_path = Path(YOLO_PRED_OUT_PATH)
if not pred_path.parent.exists():
    pred_path.parent.mkdir(parents=True, exist_ok=True)
    
df_clean.to_pickle(YOLO_PRED_OUT_PATH)

In [ ]:
df = pd.read_pickle(YOLO_PRED_OUT_PATH)
# display(df)
pred_is_worm_df = df[df['yolo_is_worm'] == True].copy()
display(pred_is_worm_df)
# display(is_worm_df.columns)

In [ ]:

output_path = Path(OUTPUT_DIR)
if not output_path.exists():
    output_path.mkdir(parents=True, exist_ok=True)
    
src_dest_dir = output_path / 'src_npy'
mask_dest_dir = output_path / 'mask_npy'
src_dest_dir.mkdir(exist_ok=True)
mask_dest_dir.mkdir(exist_ok=True)

for src, mask in zip(pred_is_worm_df['src_npy_path'], pred_is_worm_df['mask_npy_path']):
    if isinstance(src, str) and os.path.exists(src):
        shutil.copy(src, src_dest_dir)

    if isinstance(mask, str) and os.path.exists(mask):
        shutil.copy(mask, mask_dest_dir)